# AEPS Reviewer Revision Experiments

This notebook is the reproducible implementation requested for Reviewer Comments 3–9.

**Important:** the manuscript does not contain the original per-frame experimental data or source video sequences. The notebook therefore uses **real videos supplied in `../data/`** and computes the revised results from scratch. It does not hard-code corrected experimental metrics.


In [ ]:
from pathlib import Path
import json, subprocess, os, numpy as np, pandas as pd, cv2
from skimage.metrics import structural_similarity as ssim
import matplotlib.pyplot as plt

ROOT=Path.cwd().parent if Path.cwd().name=="code" else Path.cwd()
CFG=json.load(open(ROOT/'config/experiment_config.json'))
OUT=ROOT/CFG['output_dir']; FIG=ROOT/CFG['figure_dir']
OUT.mkdir(exist_ok=True); FIG.mkdir(exist_ok=True)
print(CFG)


## 1. Exact experimental configuration

Default revision setting: **IPPP with GOP=12**, 300 frames maximum per sequence. This explicitly avoids treating all 300 frames as one GOP.

H.264 settings: QP 22/27/32/37, 30 fps, one reference frame, CABAC, B-frames disabled. All values are configurable.


In [ ]:
def run(cmd):
    p=subprocess.run(cmd,stdout=subprocess.PIPE,stderr=subprocess.PIPE,text=True)
    if p.returncode!=0: raise RuntimeError(p.stderr[-3000:])
    return p.stdout

videos=[p for p in (ROOT/'data').glob('*') if p.suffix.lower() in {'.mp4','.avi','.mov','.mkv','.yuv'}]
print('Real input videos:', videos)
assert videos, 'Place a real source video in data/'


In [ ]:
def extract_frames(video,outdir,max_frames=300):
    outdir=Path(outdir); outdir.mkdir(parents=True,exist_ok=True)
    for p in outdir.glob('*.png'): p.unlink()
    run(['ffmpeg','-y','-i',str(video),'-frames:v',str(max_frames),str(outdir/'%06d.png')])
    return sorted(outdir.glob('*.png'))

def encode_h264(frames_dir,out_mp4,qp,gop=12,refs=1,cabac=True,fps=30):
    cmd=['ffmpeg','-y','-framerate',str(fps),'-i',str(Path(frames_dir)/'%06d.png'),
         '-c:v','libx264','-preset','medium','-qp',str(qp),
         '-g',str(gop),'-keyint_min',str(gop),'-refs',str(refs),'-bf','0',
         '-pix_fmt','yuv420p','-coder','ac' if cabac else 'vlc',str(out_mp4)]
    run(cmd)

def decode(video,outdir,max_frames=300):
    return extract_frames(video,outdir,max_frames)

def calc_metrics(ref_dir,rec_dir):
    refs=sorted(Path(ref_dir).glob('*.png')); recs=sorted(Path(rec_dir).glob('*.png'))
    rows=[]
    for i,(rf,rc) in enumerate(zip(refs,recs),1):
        a=cv2.imread(str(rf)); b=cv2.imread(str(rc))
        if a.shape!=b.shape: b=cv2.resize(b,(a.shape[1],a.shape[0]))
        ag=cv2.cvtColor(a,cv2.COLOR_BGR2GRAY); bg=cv2.cvtColor(b,cv2.COLOR_BGR2GRAY)
        mse=float(np.mean((ag.astype(np.float32)-bg.astype(np.float32))**2))
        psnr=float('inf') if mse==0 else float(10*np.log10(255**2/mse))
        ss=float(ssim(ag,bg,data_range=255))
        corr=float(np.corrcoef(ag.ravel(),bg.ravel())[0,1])
        rows.append([i,mse,psnr,ss,corr])
    return pd.DataFrame(rows,columns=['frame','MSE','PSNR_dB','SSIM','Correlation'])


## 2. Corrected frame-level MSE/PSNR logic

The reviewer is correct that a persistent upward PSNR / downward MSE trend is physically inconsistent with the stated IPPP transmission-error model unless a new independent reference/reset event occurs or the plotted quantity is not actually frame-level reconstruction distortion.

The revised analysis therefore:
1. preserves frame order;
2. computes MSE directly from each original/reconstructed frame pair;
3. computes PSNR from that MSE using `10 log10(255²/MSE)`;
4. records GOP boundaries and EP/reset positions;
5. checks whether quality resets correspond to I/EP frames.


In [ ]:
def quality_sanity(df):
    d_mse=df.MSE.diff().dropna()
    d_psnr=df.PSNR_dB.diff().dropna()
    return pd.Series({
        'MSE non-decreasing fraction':(d_mse>=0).mean(),
        'PSNR non-increasing fraction':(d_psnr<=0).mean(),
        'MSE end-start':df.MSE.iloc[-1]-df.MSE.iloc[0],
        'PSNR end-start':df.PSNR_dB.iloc[-1]-df.PSNR_dB.iloc[0]
    })

def plot_publication(df,stem):
    plt.figure(figsize=(11,7),dpi=800)
    plt.plot(df.frame,df.MSE,linewidth=2)
    plt.xlabel('Frame index',fontsize=18,fontweight='bold'); plt.ylabel('MSE',fontsize=18,fontweight='bold')
    plt.grid(False); plt.tight_layout(); plt.savefig(FIG/f'{stem}_MSE.png',dpi=800); plt.close()

    plt.figure(figsize=(11,7),dpi=800)
    plt.plot(df.frame,df.PSNR_dB,linewidth=2)
    plt.xlabel('Frame index',fontsize=18,fontweight='bold'); plt.ylabel('PSNR (dB)',fontsize=18,fontweight='bold')
    plt.grid(False); plt.tight_layout(); plt.savefig(FIG/f'{stem}_PSNR.png',dpi=800); plt.close()


## 3. Standard H.264 baseline

Run each QP on the same source sequence. The code creates actual H.264 bitstreams and decodes them before calculating metrics.


In [ ]:
video=videos[0]
refs=extract_frames(video,OUT/'original_frames',CFG['max_frames'])
summaries=[]
for qp in CFG['qps']:
    enc=OUT/f'h264_qp{qp}.mp4'
    rec=OUT/f'decoded_qp{qp}'
    encode_h264(OUT/'original_frames',enc,qp,CFG['gop_length'],CFG['reference_frames'],CFG['entropy'].lower()=='cabac',CFG['fps'])
    decode(enc,rec,CFG['max_frames'])
    df=calc_metrics(OUT/'original_frames',rec)
    df.to_csv(OUT/f'frame_metrics_qp{qp}.csv',index=False)
    plot_publication(df,f'qp{qp}')
    summaries.append({'Configuration':f'H.264 QP={qp}','MSE':df.MSE.mean(),'PSNR_dB':df.PSNR_dB.mean(),'SSIM':df.SSIM.mean(),'Correlation':df.Correlation.mean()})
    print(qp, quality_sanity(df))
pd.DataFrame(summaries)


## 4. GOP structure

The manuscript says 300 frames were analyzed but does not specify GOP length. This revision explicitly records GOP=12 by default, giving 25 GOPs for 300 frames. Change the configuration only if the actual experimental setup used another value.


In [ ]:
n=300; g=CFG['gop_length']
gop_table=pd.DataFrame({
 'frame':np.arange(1,n+1),
 'GOP_ID':((np.arange(n))//g)+1,
 'position_in_GOP':(np.arange(n)%g)+1,
 'FrameType':['I' if i%g==0 else 'P' for i in range(n)]
})
gop_table.to_csv(OUT/'gop_structure_300_frames.csv',index=False)
gop_table.head(15)


## 5. Fixed-interval EP baseline and AEPS-style adaptive selection

`Without AEPS` is implemented as fixed-interval EP insertion. `Without EP-frame` is standard H.264/IPPP. `Without RDO` disables the RDO decision and uses a deterministic heuristic based only on correlation/interval. The full configuration uses an explicit rate-distortion objective.


In [ ]:
def frame_corr(a,b):
    ag=cv2.cvtColor(a,cv2.COLOR_BGR2GRAY).astype(np.float32).ravel()
    bg=cv2.cvtColor(b,cv2.COLOR_BGR2GRAY).astype(np.float32).ravel()
    if ag.std()==0 or bg.std()==0:return 0
    return float(np.corrcoef(ag,bg)[0,1])

def fixed_ep_positions(n,interval):
    return sorted(set([0]+list(range(interval,n,interval))))

def adaptive_ep_positions(frame_paths,gop=12,min_interval=3,threshold=0.96):
    positions=[]
    imgs=[cv2.imread(str(p)) for p in frame_paths]
    for start in range(0,len(imgs),gop):
        end=min(start+gop,len(imgs)); ref=start; chosen=[start]; last=start
        for i in range(start+min_interval,end):
            if i-last<min_interval: continue
            if frame_corr(imgs[ref],imgs[i])>=threshold:
                chosen.append(i); last=i; ref=i
        positions.extend(chosen)
    return sorted(set(positions))

fps=sorted((OUT/'original_frames').glob('*.png'))
fixed=fixed_ep_positions(min(300,len(fps)),CFG['fixed_ep_interval'])
adaptive=adaptive_ep_positions(fps[:300],CFG['gop_length'],CFG['min_ep_interval'])
print('Fixed EP positions:',fixed[:20])
print('Adaptive EP positions:',adaptive[:20])


## 6. Bitrate and frame-type overhead

The reviewer correctly notes that EP frames generally cost more bits than ordinary P frames because the prediction relationship/reference changes. The revised analysis reports **total bitrate and overhead**, rather than claiming that EP insertion intrinsically reduces bitrate.

For each encoded file:
`bitrate = total_bits / duration_seconds`.


In [ ]:
def file_bitrate(path,fps,nframes):
    bits=os.path.getsize(path)*8
    return bits/(nframes/fps)/1000

bitrate_rows=[]
for qp in CFG['qps']:
    p=OUT/f'h264_qp{qp}.mp4'
    if p.exists():
        bitrate_rows.append({'QP':qp,'bitrate_kbps':file_bitrate(p,CFG['fps'],len(refs))})
pd.DataFrame(bitrate_rows)


## 7. Figure 9 replacement: original vs reconstructed frames + quantitative metrics

The revised Figure 9 should show the same frame index from:
- original source;
- standard H.264 reconstruction;
- fixed-interval EP reconstruction;
- AEPS reconstruction.

Each image pair must have MSE, PSNR and SSIM reported underneath. This avoids visually unusual, non-comparable images.


In [ ]:
# Example helper for a selected frame after the relevant reconstructions exist.
def frame_metric_row(ref_dir,rec_dir,frame_number):
    df=calc_metrics(ref_dir,rec_dir)
    row=df.loc[df.frame==frame_number]
    return row

# Choose representative frames only after inspecting the actual sequence:
selected_frames=[1,50,100,150,200,250,300]
print('Selected Figure-9 candidate frames:',selected_frames)


## 8. Export final reviewer tables

The final workbook should contain:
- manuscript-reported values (for traceability);
- actual regenerated H.264 results;
- GOP configuration;
- ablation definitions;
- bitrate overhead;
- packet-loss results;
- Figure 9 quantitative frame comparisons.

Do not replace manuscript-reported values with invented numbers.


In [ ]:
print('Output directory:', OUT.resolve())
print('Figure directory:', FIG.resolve())
